# Paper-Aligned Elevation Partitioning + YOLO11s-OBB

This notebook follows the **dataset partition protocol** reported for
Pol-ISARSpaceTarget-1.0: `D40`, `D45`, `D50`, and `Dall`, each retaining the
author's official 8:1:1 train/validation/test membership. It then performs a
controlled single-VV YOLO11s-OBB study: four training runs and twelve
cross-elevation test evaluations.

This is **not** a direct reproduction of the paper's eight-channel MMRotate
detectors. The paper protocol is used for partitioning; the detector and input
representation are deliberately held to the current YOLO baseline.

## 1. Colab setup

In [ ]:
get_ipython().run_line_magic(
    "pip",
    'install -q -U "Pillow>=10.4,<12" "ultralytics>=8.3,<9" "PyYAML>=6" "seaborn>=0.13"',
)

In [ ]:
import gc
import json
import os
import shutil
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from PIL import Image
from ultralytics import YOLO

from google.colab import drive

print("Python:", sys.version)
print("Pillow:", Image.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is not enabled. In Colab select Runtime > Change runtime type > T4 GPU.")

## 2. Experiment configuration

Use `SMOKE_TEST=True` first. A smoke run builds at most 12 records per split
and trains for 2 epochs. Then set it to `False`, restart, and run all cells.

In [ ]:
MODEL_WEIGHTS = "yolo11s-obb.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 8
SEED = 42
PATIENCE = 20
WORKERS = 2

RUN_TRAINING = True
RUN_EVALUATION = True
SMOKE_TEST = False
SMOKE_MAX_PER_SPLIT = 12
FORCE_REBUILD = False

if SMOKE_TEST:
    EPOCHS = 2
    PATIENCE = 2

EXPERIMENT_TAG = "smoke" if SMOKE_TEST else "full100"
print({"model": MODEL_WEIGHTS, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "tag": EXPERIMENT_TAG})

## 3. Mount Drive and locate the dataset/helper

In [ ]:
MOUNT = Path("/content/drive")
if not (MOUNT / "MyDrive").exists():
    drive.mount(str(MOUNT))
else:
    print("Drive already mounted:", MOUNT)

DRIVE_BASE = MOUNT / "MyDrive"
DRDO_DIR = DRIVE_BASE / "drdo"

candidate_roots = [
    DRDO_DIR / "Pol_ISARSpaceTarget1.0",
    DRDO_DIR / "Pol-ISARSpaceTarget1.0",
]
DATASET_ROOT = next(
    (root for root in candidate_roots if (root / "ImageSets" / "labelTxt").exists()),
    None,
)
if DATASET_ROOT is None:
    matches = [p.parent for p in DRDO_DIR.rglob("ImageSets") if (p / "labelTxt").exists() and (p / "images").exists()]
    DATASET_ROOT = matches[0] if matches else None
if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Dataset not found. Expected /content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0/"
        " with ImageSets/labelTxt and ImageSets/images."
    )

# The supplied package is authoritative so an older helper elsewhere on Drive
# cannot silently change the experiment.
package_zip = DRDO_DIR / "POLISAR_PaperPartition_YOLO_OBB_PACKAGE.zip"
helper_candidates = []
if package_zip.exists():
    package_root = Path("/content/POLISAR_PaperPartition_YOLO_OBB_PACKAGE")
    if package_root.exists():
        shutil.rmtree(package_root)
    shutil.unpack_archive(package_zip, package_root)
    helper_candidates = list(package_root.rglob("paper_elevation_partition.py"))
if not helper_candidates:
    helper_candidates = list(DRDO_DIR.rglob("paper_elevation_partition.py"))
if not helper_candidates:
    raise FileNotFoundError(
        "paper_elevation_partition.py was not found in MyDrive/drdo. Upload the supplied "
        "POLISAR_PaperPartition_YOLO_OBB_PACKAGE.zip, extract it under MyDrive/drdo, and rerun this cell."
    )
HELPER_DIR = helper_candidates[0].parent.parent
sys.path.insert(0, str(HELPER_DIR))

from src.paper_elevation_partition import (
    ELEVATIONS,
    OFFICIAL_SPLITS,
    audit_partitions,
    build_yolo_partition,
    discover_frames,
    partition_records,
    read_dota_rows,
    write_partition_manifests,
)

TIFF_ROOT = DATASET_ROOT / "TIFFImages" / "AmplitudeImages"
LOCAL_DATASETS = Path("/content/polisar_paper_partitions") / EXPERIMENT_TAG
DRIVE_OUT = DRDO_DIR / "paper_partition_yolo_obb" / EXPERIMENT_TAG
MANIFEST_DIR = DRIVE_OUT / "manifests"
RUNS_DIR = DRIVE_OUT / "runs"
PLOTS_DIR = DRIVE_OUT / "plots"
for path in (MANIFEST_DIR, RUNS_DIR, PLOTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_ROOT)
print("Helper:", helper_candidates[0])
print("Drive output:", DRIVE_OUT)

## 4. Create and audit the paper partitions

The official split membership is preserved. We only filter it by elevation;
no new random split is generated. Full-data expected counts are:

| Partition | Train | Validation | Test |
|---|---:|---:|---:|
| D40 / D45 / D50 | 438 | 54 | 54 |
| Dall | 1,314 | 162 | 162 |

In [ ]:
records = discover_frames(DATASET_ROOT)
partitions = partition_records(records)

if SMOKE_TEST:
    partitions = {
        name: {split: rows[:SMOKE_MAX_PER_SPLIT] for split, rows in split_map.items()}
        for name, split_map in partitions.items()
    }
    audit = audit_partitions(partitions, strict_counts=False)
else:
    audit = audit_partitions(partitions, strict_counts=True)

write_partition_manifests(records, partitions, MANIFEST_DIR)
count_table = pd.DataFrame(audit["counts"]).T[["train", "valid", "test"]]
display(count_table)

master = pd.DataFrame([row.to_dict() for row in records])
display(pd.crosstab([master.satellite, master.elevation], master.official_split))
azimuth_summary = master.groupby(["satellite", "elevation", "official_split"]).azimuth.agg(["count", "min", "max"])
display(azimuth_summary)

split_sets = {
    split: set(master.loc[master.official_split == split, "base"])
    for split in OFFICIAL_SPLITS
}
overlap_report = {
    "train_valid": len(split_sets["train"] & split_sets["valid"]),
    "train_test": len(split_sets["train"] & split_sets["test"]),
    "valid_test": len(split_sets["valid"] & split_sets["test"]),
}
print("Split overlap audit (all must be zero):", overlap_report)

## 5. Derive the component class map

In [ ]:
class_names = sorted({
    class_name
    for record in records
    for _, class_name, _ in read_dota_rows(record.label_path)
})
class_to_id = {name: index for index, name in enumerate(class_names)}
if not class_to_id:
    raise ValueError("No component classes were found in the DOTA annotations.")
print("Class map:", class_to_id)
(MANIFEST_DIR / "class_to_id.json").write_text(json.dumps(class_to_id, indent=2), encoding="utf-8")

component_rows = []
for record in records:
    for _, class_name, difficult in read_dota_rows(record.label_path):
        component_rows.append({"split": record.official_split, "class": class_name, "difficult": difficult})
component_counts = pd.DataFrame(component_rows).groupby(["split", "class"]).size().unstack(fill_value=0)
display(component_counts)
component_counts.to_csv(MANIFEST_DIR / "component_class_counts.csv")

## 6. Build four RAM-safe YOLO datasets

Each frame is loaded, converted, written, and released before the next frame.
A completion manifest prevents incomplete datasets from being reused.

In [ ]:
dataset_yamls = {}
for partition_name in ("D40", "D45", "D50", "Dall"):
    print("Building", partition_name)
    dataset_yamls[partition_name] = build_yolo_partition(
        partition_name,
        partitions[partition_name],
        LOCAL_DATASETS,
        TIFF_ROOT,
        class_to_id,
        force_rebuild=FORCE_REBUILD,
    )
    shutil.copy2(
        dataset_yamls[partition_name].parent / "build_complete.json",
        MANIFEST_DIR / f"{partition_name}_build_complete.json",
    )
    gc.collect()

print({name: str(path) for name, path in dataset_yamls.items()})

## 7. Visual audit: one frame per elevation

In [ ]:
def draw_yolo_obb(ax, image_path, label_path, names):
    image = np.asarray(Image.open(image_path).convert("RGB"))
    height, width = image.shape[:2]
    ax.imshow(image)
    for line in label_path.read_text(encoding="utf-8").splitlines():
        values = line.split()
        class_id = int(values[0])
        points = np.array([float(v) for v in values[1:]], dtype=float).reshape(4, 2)
        points[:, 0] *= width
        points[:, 1] *= height
        closed = np.vstack([points, points[0]])
        ax.plot(closed[:, 0], closed[:, 1], linewidth=1.5, label=names[class_id])
    ax.axis("off")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, elevation in zip(axes, ELEVATIONS):
    dataset_dir = dataset_yamls[f"D{elevation}"].parent
    image_path = next((dataset_dir / "images" / "train").glob("*.png"))
    label_path = dataset_dir / "labels" / "train" / f"{image_path.stem}.txt"
    draw_yolo_obb(ax, image_path, label_path, {v: k for k, v in class_to_id.items()})
    ax.set_title(f"Elevation {elevation} degrees: {image_path.stem}")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "partition_samples_with_obb.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Train four models

One fresh YOLO11s-OBB model is trained on each of `D40`, `D45`, `D50`, and
`Dall`. Checkpoint selection uses that partition's official validation set.
Test sets are not inspected during training.

In [ ]:
checkpoints = {}
for train_partition in ("D40", "D45", "D50", "Dall"):
    run_name = f"{MODEL_WEIGHTS.replace('.pt', '')}_{train_partition}_{EXPERIMENT_TAG}_seed{SEED}"
    run_dir = RUNS_DIR / run_name
    best_path = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"
    if best_path.exists() and results_csv.exists():
        print("Resume: using completed run", run_dir)
    elif RUN_TRAINING:
        model = YOLO(MODEL_WEIGHTS)
        model.train(
            data=str(dataset_yamls[train_partition]),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            seed=SEED,
            patience=PATIENCE,
            workers=WORKERS,
            cache=False,
            device=0,
            project=str(RUNS_DIR),
            name=run_name,
            exist_ok=True,
            plots=True,
            verbose=True,
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()
    if not best_path.exists():
        raise FileNotFoundError(f"Best checkpoint missing after {train_partition}: {best_path}")
    checkpoints[train_partition] = best_path

## 9. Twelve-condition cross-elevation evaluation

Four trained checkpoints are each evaluated on the official test sets of
`D40`, `D45`, and `D50`: 4 x 3 = 12 test conditions.

In [ ]:
def metric_value(metric_object, name):
    value = getattr(metric_object, name)
    return float(value() if callable(value) else value)

evaluation_csv = DRIVE_OUT / "paper_partition_yolo_12_conditions.csv"
evaluation_json = DRIVE_OUT / "paper_partition_yolo_12_conditions.json"

if RUN_EVALUATION:
    evaluation_rows = []
    for train_partition, checkpoint in checkpoints.items():
        model = YOLO(str(checkpoint))
        for test_elevation in ELEVATIONS:
            test_partition = f"D{test_elevation}"
            started = time.time()
            metrics = model.val(
                data=str(dataset_yamls[test_partition]),
                split="test",
                imgsz=IMGSZ,
                batch=BATCH,
                workers=WORKERS,
                device=0,
                plots=False,
                verbose=False,
            )
            obb = getattr(metrics, "obb", None) or metrics.box
            precision = metric_value(obb, "mp")
            recall = metric_value(obb, "mr")
            evaluation_rows.append({
                "train_partition": train_partition,
                "test_elevation": test_elevation,
                "precision": precision,
                "recall": recall,
                "f1": 2 * precision * recall / max(precision + recall, 1e-12),
                "mAP50": metric_value(obb, "map50"),
                "mAP50_95": metric_value(obb, "map"),
                "elapsed_min": (time.time() - started) / 60,
                "checkpoint": str(checkpoint),
            })
        del model
        gc.collect()
        torch.cuda.empty_cache()
    evaluation = pd.DataFrame(evaluation_rows)
    evaluation.to_csv(evaluation_csv, index=False)
    evaluation_json.write_text(evaluation.to_json(orient="records", indent=2), encoding="utf-8")
else:
    evaluation = pd.read_csv(evaluation_csv)

display(evaluation)

## 10. Comparative metrics and capacity plots

In [ ]:
metric_columns = ["precision", "recall", "f1", "mAP50", "mAP50_95"]
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for ax, metric in zip(axes.flat, metric_columns):
    matrix = evaluation.pivot(index="train_partition", columns="test_elevation", values=metric)
    matrix = matrix.reindex(["D40", "D45", "D50", "Dall"])
    sns.heatmap(matrix, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="viridis", ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("Test elevation")
    ax.set_ylabel("Training partition")
axes.flat[-1].axis("off")
plt.suptitle("YOLO11s-OBB cross-elevation generalization (official test sets)", y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "cross_elevation_metric_heatmaps.png", dpi=180, bbox_inches="tight")
plt.show()

def transfer_condition(row):
    if row.train_partition == "Dall":
        return "Dall model"
    return "same elevation" if int(row.train_partition[1:]) == row.test_elevation else "unseen elevation"

evaluation["condition"] = evaluation.apply(transfer_condition, axis=1)
display(evaluation.groupby("condition")[metric_columns].agg(["mean", "std", "count"]))

In [ ]:
loss_candidates = {
    "train/box_loss": "train box",
    "val/box_loss": "validation box",
    "train/cls_loss": "train class",
    "val/cls_loss": "validation class",
    "train/dfl_loss": "train DFL",
    "val/dfl_loss": "validation DFL",
}
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
for ax, train_partition in zip(axes.flat, ("D40", "D45", "D50", "Dall")):
    results_path = checkpoints[train_partition].parents[1] / "results.csv"
    history = pd.read_csv(results_path)
    history.columns = [column.strip() for column in history.columns]
    epoch = history.get("epoch", pd.Series(np.arange(1, len(history) + 1)))
    for column, label in loss_candidates.items():
        if column in history:
            ax.plot(epoch, history[column], label=label)
    ax.set_title(f"{train_partition}: training and validation losses")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "four_model_train_validation_losses.png", dpi=180, bbox_inches="tight")
plt.show()

## 11. Interpretation and limitations

- Compare the diagonal `D40->40`, `D45->45`, and `D50->50` cells with the
  off-diagonal cells to measure elevation transfer.
- Compare `Dall` with the three elevation-specific models to test whether
  multi-elevation training improves robustness.
- Use mAP50-95 as the stricter localization result; precision, recall, F1 and
  mAP50 provide supporting views.
- These are simulated views of the same six satellite identities. Strong
  cross-elevation scores do not establish transfer to unseen satellite types,
  sensors, noise regimes, or measured operational imagery.
- This notebook uses a three-channel single-VV magnitude image. The paper's
  eight real/imaginary polarimetric channels and MMRotate detectors remain a
  separate experiment and must not be conflated with these results.
- Validation selects `best.pt`; test results are used only for final reporting.

In [ ]:
print("Completed outputs:")
print("  12-condition table:", evaluation_csv)
print("  plots:", PLOTS_DIR)
print("  checkpoints:", RUNS_DIR)